In [15]:
from neo4j import GraphDatabase
import spacy
import pandas as pd
from typing import List, Dict, Tuple
import re
# Carregue o modelo NLP em português
nlp = spacy.load("pt_core_news_sm")

In [16]:
user = "neo4j"
password = "neo4jneo4j"
uri = "bolt://localhost:7687"

# Parte 1 – Conexão ao Neo4j

## Questão 1.1. Configure a conexão ao Neo4j.

In [17]:
class Neo4jConnection:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self):
        self.driver.close()

    def query(self, query, parameters=None):
        with self.driver.session() as session:
            result = session.run(query, parameters or {})
            return [record.data() for record in result]


conn = Neo4jConnection(uri, user, password)
result = conn.query("RETURN 'Conexão bem-sucedida!' as message")
print(result[0]["message"])

Conexão bem-sucedida!


# Parte 2 – NLP para Extração de Entidades

## Questão 2.1. Extraia entidades com spaCy

In [18]:
def extract_entities(text: str) -> List[Tuple[str, str]]:
    """
    Retorne lista de tuplas (entidade, tipo)
    """
    doc = nlp(text)
    entities = []
    for ent in doc.ents:
        entities.append((ent.text, ent.label_))
    return entities

# Texto de exemplo
texto_exemplo = """
A OpenAI lançou o ChatGPT em novembro de 2022. Sam Altman criou a empresa.
O ChatGPT utiliza aprendizagem profunda e foi desenvolvido em São Francisco.
"""

entities = extract_entities(texto_exemplo)
print("Entidades encontradas:")
for entity, label in entities:
    print(f"- {entity} ({label})")

Entidades encontradas:
- OpenAI (ORG)
- ChatGPT (MISC)
- Sam Altman (PER)
- ChatGPT (MISC)
- São Francisco (LOC)


## Questão 2.2: Extração de relações simples.

In [19]:
def extract_relations(text: str) -> List[Tuple[str, str, str]]:
    """
    Extrai relações simples baseadas em padrões sintáticos.
    Retorna lista de triplas (sujeito, relação, objeto)
    """
    doc = nlp(text)
    relations = []
    for token in doc:
        if token.dep_ == "nsubj":
            verb = token.head
            for child in verb.children:
                if child.dep_ == "obj":
                    relations.append((token.text, verb.lemma_, child.text))
    return relations

relations = extract_relations(texto_exemplo)
print("\nRelações encontradas:")
for subj, rel, obj in relations:
    print(f"- {subj} --[{rel}]--> {obj}")


Relações encontradas:
- OpenAI --[lançar]--> ChatGPT
- Sam --[criar]--> empresa
- ChatGPT --[utilizar]--> aprendizagem


## Questão 2.3. Quais são as limitações desta abordagem baseada apenas em dependências sintáticas? Como poderíamos melhorá-la?

### Limitações:
- Só apanha padrões simples (ex.: nsubj + verbo + obj); falha em frases complexas.
- Erros de POS tagging ou parsing propagam-se diretamente para relações erradas.
- Usa tokens isolados em vez de expressões completas, utilizando entidades incompletas (ex.: “Sam Altman” vs “Altman”).
- Não resolve homónimos nem contexto (ex.: “Apple” empresa vs fruta).
- Modelos genéricos como o spaCy degradam em textos técnicos, com abreviações ou com neologismos.
- A mesma relação pode ser extraída várias vezes ou com variantes inconsistentes.

### Como melhorar
- Combinar NER e sintaxe ao alinhar sujeito/objeto a entidades nomeadas para triplas mais limpas.
- Usar padrões Cypher/regex para verbos e construções frequentes no corpus.
- Usar modelos de RE supervisionados (OpenIE, REBEL) ou fine-tuning de transformers para extração de relações.
- Deduplicar triplas, mapear sinónimos/lemas a um vocabulário de relações (ex.: criar → FOUNDED).
- Descartar relações com parser de baixa confiança ou sem entidades reconhecidas.
- No Neo4j, fundir nós por nome normalizado e rever manualmente relações de baixa qualidade.

# Parte 3: Construção do Knowledge Graph

## Questão 3.1. Criação de Nós e Relacionamentos

In [20]:
def create_knowledge_graph(conn: Neo4jConnection, entities: List[Tuple[str, str]], relations: List[Tuple[str, str, str]]):
    """
    Crie nós e relacionamentos no Neo4j baseado nas entidades e relações extraídas
    """
    conn.query("""
        CREATE CONSTRAINT entity_name_unique IF NOT EXISTS
        FOR (e:Entity) REQUIRE e.name IS UNIQUE
    """)

    for name, entity_type in entities:
        conn.query(
            """
            MERGE (e:Entity {name: $name})
            SET e.type = $type
            """,
            {"name": name, "type": entity_type},
        )

    for subject, relation, obj in relations:
        # Change ç for c
        relation = relation.replace("ç", "c")
        rel_type = re.sub(r"[^A-Z0-9_]", "_", relation.upper())
        if not rel_type or rel_type[0].isdigit():
            rel_type = f"REL_{rel_type}" if rel_type else "RELATION"
        conn.query(
            f"""
            MATCH (s:Entity {{name: $subject}})
            MATCH (o:Entity {{name: $object}})
            MERGE (s)-[:{rel_type}]->(o)
            """,
            {"subject": subject, "object": obj},
        )


create_knowledge_graph(conn, entities, relations)
print("Knowledge graph criado com sucesso!")

Knowledge graph criado com sucesso!


## Questão 3.2. Consultas Cypher através do Python

In [21]:
def query_graph(conn: Neo4jConnection, entity_name: str):
    """
    Consulte o knowledge graph para encontrar conexões de uma entidade
    """
    query = """
    MATCH (e:Entity {name: $entity_name})
    OPTIONAL MATCH (e)-[r]-(connected)
    WITH e,
         [n IN collect(DISTINCT connected.name) WHERE n IS NOT NULL] AS connections,
         [t IN collect(DISTINCT type(r)) WHERE t IS NOT NULL] AS relation_types
    RETURN e.name AS entity, connections, relation_types
    """
    result = conn.query(query, {"entity_name": entity_name})
    return result[0] if result else None

info = query_graph(conn, "OpenAI")
if info:
    print(f"\nEntidade: {info['entity']}")
    print(f"Conexões: {info['connections']}")
    print(f"Tipos de relação: {info['relation_types']}")


Entidade: OpenAI
Conexões: ['ChatGPT']
Tipos de relação: ['LANCAR']


## Questão 3.3. Porque usamos MERGE em vez de CREATE ao criar os nós? Qual a importância das constraints?

MERGE cria o nó só se ainda não existir; se já existir, reutiliza-o. Assim evitamos duplicados ao correr o código várias vezes ou quando a mesma entidade aparece mais do que uma vez.

CREATE cria sempre um nó novo, mesmo que já exista um igual, então o grafo ficaria com entradas repetidas.

A constraint de unicidade no nome garante ao nível da base de dados que não podem existir dois nós Entity com o mesmo nome. Isso reforça o MERGE e evita duplicados mesmo que haja erros na lógica da aplicação.

# Parte 4 - Processamento de Múltiplos Documentos

## Questão 4.1. Implemente o seguinte bloco de código para processamento em lote.

In [22]:
# Documentos de exemplo
documentos = [
    {
    "id": 1,
    "texto": "Elon Musk fundou a SpaceX em 2002. A empresa lançou o Falcon 9.",
    "fonte": "Wikipedia"
    },
    {
    "id": 2,
    "texto": "Tesla, liderada por Elon Musk, produz veículos elétricos.",
    "fonte": "Notícia"
    },
    {
    "id": 3,
    "texto": "SpaceX e Tesla colaboram em projetos de tecnologia sustentável.",
    "fonte": "Artigo"
    }
]

In [ ]:
def process_documents(conn: Neo4jConnection, documents: List[Dict]):
    """
    Processa múltiplos documentos e enriquece o knowledge graph
    """
    for doc in documents:
        texto = doc["texto"]
        doc_entities = extract_entities(texto)
        doc_relations = extract_relations(texto)

        create_knowledge_graph(conn, doc_entities, doc_relations)

        conn.query(
            """
            MERGE (d:Document {id: $id})
            SET d.source = $source, d.text = $text
            """,
            {"id": doc["id"], "source": doc["fonte"], "text": texto},
        )

        for name, _ in doc_entities:
            conn.query(
                """
                MATCH (e:Entity {name: $name})
                MATCH (d:Document {id: $id})
                MERGE (e)-[:MENTIONED_IN]->(d)
                """,
                {"name": name, "id": doc["id"]},
            )


process_documents(conn, documentos)
print("Documentos processados e integrados ao grafo!")

Documentos processados e integrados ao grafo!


# Questão 4.2. Análise de Centralidade

In [24]:
def find_important_entities(conn: Neo4jConnection):
    """
    Encontra as entidades mais conectadas no grafo
    """
    query = """
    MATCH (e:Entity)
    OPTIONAL MATCH (e)-[]-(other:Entity)
    WITH e, count(DISTINCT other) AS connections
    RETURN e.name AS name, e.type AS type, connections
    ORDER BY connections DESC
    LIMIT 5
    """
    results = conn.query(query)
    for row in results:
        print(f"{row['name']} ({row['type']}): {row['connections']} conexões")
    return results


find_important_entities(conn)

OpenAI (ORG): 1 conexões
ChatGPT (MISC): 1 conexões
Sam Altman (PER): 0 conexões
Elon Musk (PER): 0 conexões
São Francisco (LOC): 0 conexões


[{'name': 'OpenAI', 'type': 'ORG', 'connections': 1},
 {'name': 'ChatGPT', 'type': 'MISC', 'connections': 1},
 {'name': 'Sam Altman', 'type': 'PER', 'connections': 0},
 {'name': 'Elon Musk', 'type': 'PER', 'connections': 0},
 {'name': 'São Francisco', 'type': 'LOC', 'connections': 0}]